In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
rng = np.random.default_rng(42)
import ipywidgets as widgets
from ipywidgets import interact, interactive_output
from IPython.display import display, Markdown

# Semana 4A — Muestreo y Estimación (Teoría)

**Objetivo:** Construir la intuición sobre por qué la media muestral es
un buen estimador y bajo qué condiciones, usando simulaciones interactivas
con datos sintéticos.

**Temas:**
1. Población vs muestra
2. Estadísticos muestrales
3. Ley de Grandes Números (LGN)
4. Teorema del Límite Central (TLC)

---
## 1. Población vs muestra

- **Población:** El conjunto *completo* de valores que nos interesan.
  En la práctica casi nunca la observamos entera.
- **Muestra:** Un subconjunto de la población que sí observamos.
  A partir de ella *estimamos* propiedades de la población.

Usaremos una población conocida $\mathcal{N}(\mu=25,\;\sigma=5)$
para poder comparar los estimados muestrales con los parámetros reales.

### Parámetros de la simulación

In [ ]:
n_muestras_slider = widgets.IntSlider(min=5, max=50, step=5, value=20, description="Número de muestras")
n_obs_slider = widgets.IntSlider(min=5, max=200, step=5, value=10, description="Tamaño de cada muestra (n)",
)

def _update(n_muestras_slider, n_obs_slider):
    _mu, _sigma = 25.0, 5.0
    _n = n_obs_slider
    _k = n_muestras_slider

    _medias = []
    for _ in range(_k):
        _muestra = rng.normal(_mu, _sigma, _n)
        _medias.append(_muestra.mean())
    _medias = np.array(_medias)

    fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(12, 4))

    # Panel izquierdo: histograma de la población
    _x_pop = np.linspace(5, 45, 300)
    _pdf_pop = (
        1
        / (_sigma * np.sqrt(2 * np.pi))
        * np.exp(-0.5 * ((_x_pop - _mu) / _sigma) ** 2)
    )
    ax1a.plot(_x_pop, _pdf_pop, "k-", lw=2, label="Población $N(25, 5^2)$")
    ax1a.axvline(_mu, color="crimson", ls="--", label=f"μ = {_mu}")
    ax1a.set_xlabel("Valor")
    ax1a.set_ylabel("Densidad")
    ax1a.set_title("Distribución de la Población")
    ax1a.legend()

    # Panel derecho: medias muestrales
    ax1b.scatter(range(_k), _medias, c="steelblue", edgecolors="white", zorder=3)
    ax1b.axhline(_mu, color="crimson", ls="--", label=f"μ = {_mu}")
    ax1b.set_xlabel("# de muestra")
    ax1b.set_ylabel("Media muestral $\\bar{x}$")
    ax1b.set_title(f"{_k} medias muestrales  (n = {_n})")
    ax1b.legend()

    plt.tight_layout()
    plt.show()

out = widgets.interactive_output(_update, {'n_muestras_slider': n_muestras_slider, 'n_obs_slider': n_obs_slider})
display(widgets.VBox([n_muestras_slider, n_obs_slider, out]))

> **Observa:** Cada muestra da una media *distinta*, pero todas oscilan
> alrededor del verdadero $\mu = 25$.  Al aumentar $n$ (tamaño de muestra)
> la dispersión de las medias disminuye.

---
## 2. Estadísticos muestrales

| Parámetro poblacional | Estadístico muestral | Fórmula |
|:---|:---|:---|
| Media $\mu$ | Media muestral $\bar{x}$ | $\frac{1}{n}\sum x_i$ |
| Varianza $\sigma^2$ | Varianza muestral $s^2$ | $\frac{1}{n-1}\sum (x_i - \bar{x})^2$ |

La media muestral $\bar{x}$ es un **estimador insesgado** de $\mu$:
en promedio, $E[\bar{x}] = \mu$.

Su dispersión depende de $n$:
$$\text{Var}(\bar{x}) = \frac{\sigma^2}{n}
\quad\Rightarrow\quad
\text{DE}(\bar{x}) = \frac{\sigma}{\sqrt{n}}$$

A mayor $n$, $\bar{x}$ está más cerca de $\mu$ — esto es la esencia
de la Ley de Grandes Números.

### Efecto del tamaño de muestra en la dispersión de $\bar{x}$

In [ ]:
n_estadisticos_slider = widgets.IntSlider(min=5, max=500, step=5, value=30, description="Tamaño de muestra (n)",
)

def _update(n_estadisticos_slider):
    _mu, _sigma = 25.0, 5.0
    _n = n_estadisticos_slider
    _num_replicas = 2000
    _medias = np.array(
        [rng.normal(_mu, _sigma, _n).mean() for _ in range(_num_replicas)]
    )

    _de_teorica = _sigma / np.sqrt(_n)
    _de_empirica = _medias.std()

    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.hist(_medias, bins=40, density=True, color="steelblue", alpha=0.7,
             edgecolor="white", label="Histograma de $\\bar{x}$")

    _x = np.linspace(_medias.min(), _medias.max(), 200)
    _pdf = (
        1
        / (_de_teorica * np.sqrt(2 * np.pi))
        * np.exp(-0.5 * ((_x - _mu) / _de_teorica) ** 2)
    )
    ax2.plot(_x, _pdf, "crimson", lw=2,
             label=f"$N(\\mu,\\;\\sigma^2/n)$ — DE teórica = {_de_teorica:.3f}")
    ax2.axvline(_mu, color="black", ls="--", alpha=0.5)
    ax2.set_xlabel("$\\bar{x}$")
    ax2.set_ylabel("Densidad")
    ax2.set_title(
        f"Distribución de medias muestrales  (n = {_n},  "
        f"DE empírica = {_de_empirica:.3f})"
    )
    ax2.legend()
    plt.tight_layout()
    plt.show()

out = widgets.interactive_output(_update, {'n_estadisticos_slider': n_estadisticos_slider})
display(widgets.VBox([n_estadisticos_slider, out]))

---
## 3. Ley de Grandes Números (LGN)

> Conforme el tamaño de la muestra $n \to \infty$, la media muestral
> $\bar{x}_n$ converge al valor esperado $\mu$, **sin importar la forma
> de la distribución original**.

$$\bar{x}_n = \frac{1}{n}\sum_{i=1}^{n} x_i \xrightarrow{n\to\infty} \mu$$

Vamos a verificarlo con tres distribuciones muy distintas:
- **Normal** $N(25, 5^2)$ — simétrica, bien comportada.
- **Exponencial** $\text{Exp}(\lambda=0.2)$ — asimétrica con cola derecha.  Esperanza $= 1/\lambda = 5$.
- **Bernoulli** $p = 0.3$ — discreta, solo 0 y 1.  Esperanza $= p = 0.3$.

In [ ]:
_max_n = 50_000
_normal = rng.normal(25, 5, _max_n)
_exponencial = rng.exponential(5, _max_n)
_bernoulli = rng.binomial(1, 0.3, _max_n)

_ns = np.arange(1, _max_n + 1)

fig3, axes3 = plt.subplots(1, 3, figsize=(14, 4))

_configs = [
    (_normal, 25.0, "Normal(25, 25)", "steelblue"),
    (_exponencial, 5.0, "Exp(λ=0.2)", "darkorange"),
    (_bernoulli, 0.3, "Bernoulli(0.3)", "seagreen"),
]

for ax, (datos, mu_real, nombre, color) in zip(axes3, _configs):
    _media_acum = np.cumsum(datos) / _ns
    ax.plot(_ns, _media_acum, color=color, lw=0.5)
    ax.axhline(mu_real, color="crimson", ls="--", lw=1.5,
                label=f"μ = {mu_real}")
    ax.set_xscale("log")
    ax.set_xlabel("n (escala log)")
    ax.set_ylabel("Media acumulada")
    ax.set_title(nombre)
    ax.legend()

plt.suptitle("Ley de Grandes Números — Convergencia de la media muestral",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Conclusión:** Independientemente de la distribución (normal, asimétrica,
> discreta), la media acumulada converge al verdadero $\mu$.
> Con pocas observaciones la estimación puede estar lejos;
> con miles, se estabiliza.

---
## 4. Teorema del Límite Central (TLC)

> Si tomamos muestras de tamaño $n$ de **cualquier** distribución
> con media $\mu$ y varianza $\sigma^2$ finitas,
> la distribución de la media muestral $\bar{x}$ se aproxima a una
> **normal** conforme $n$ crece:
>
> $$\bar{x} \;\dot\sim\; N\!\left(\mu,\;\frac{\sigma^2}{n}\right)$$

Esto es extraordinario: **no importa la forma de la distribución original**.
Lo verificaremos con la distribución **exponencial** (fuertemente asimétrica)
y la **uniforme** (sin colas).

### Elige la distribución para explorar el TLC

In [ ]:
dist_dropdown = widgets.Dropdown(options=["Exponencial", "Uniforme"], value="Exponencial", description="Distribución original")

def _update(dist_dropdown):
    _dist = dist_dropdown
    if _dist == "Exponencial":
        _lam = 0.5
        _mu_orig = 1 / _lam
        _sigma_orig = 1 / _lam
        _gen = lambda size: rng.exponential(1 / _lam, size)
        _dist_name = f"Exponencial(λ={_lam})"
    else:
        _a, _b = 0.0, 10.0
        _mu_orig = (_a + _b) / 2
        _sigma_orig = (_b - _a) / np.sqrt(12)
        _gen = lambda size: rng.uniform(_a, _b, size)
        _dist_name = f"Uniforme({_a:.0f}, {_b:.0f})"

    _tamaños = [2, 5, 30, 100]
    _num_replicas = 3000

    fig4, axes4 = plt.subplots(2, 2, figsize=(11, 8))

    for ax, n_val in zip(axes4.flat, _tamaños):
        _medias = np.array(
            [_gen(n_val).mean() for _ in range(_num_replicas)]
        )
        ax.hist(_medias, bins=50, density=True, color="steelblue",
                alpha=0.7, edgecolor="white")

        # Curva normal teórica
        _de = _sigma_orig / np.sqrt(n_val)
        _x = np.linspace(_medias.min(), _medias.max(), 200)
        _pdf = (
            1
            / (_de * np.sqrt(2 * np.pi))
            * np.exp(-0.5 * ((_x - _mu_orig) / _de) ** 2)
        )
        ax.plot(_x, _pdf, "crimson", lw=2, label="Normal teórica")
        ax.set_title(f"n = {n_val}")
        ax.set_xlabel("$\\bar{x}$")
        ax.set_ylabel("Densidad")
        ax.legend(fontsize=8)

    plt.suptitle(
        f"TLC — Distribución original: {_dist_name}  "
        f"(μ = {_mu_orig:.2f},  σ = {_sigma_orig:.2f})",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()

out = widgets.interactive_output(_update, {'dist_dropdown': dist_dropdown})
display(widgets.VBox([dist_dropdown, out]))

> **Observa:**
> - Con $n = 2$, la distribución de $\bar{x}$ todavía refleja la
>   asimetría o forma de la distribución original.
> - Con $n = 30$, ya es prácticamente normal.
> - Con $n = 100$, el ajuste es casi perfecto.
>
> Esto justifica el uso de la distribución normal para construir
> intervalos de confianza y pruebas de hipótesis (semanas 5 y 6),
> incluso cuando los datos originales no son normales.

---
## Resumen

| Concepto | Enunciado clave |
|:---|:---|
| **Población vs muestra** | Estimamos parámetros ($\mu$, $\sigma^2$) con estadísticos muestrales ($\bar{x}$, $s^2$) |
| **LGN** | $\bar{x}_n \to \mu$ cuando $n \to \infty$ — más datos = mejor estimación |
| **TLC** | $\bar{x} \;\dot\sim\; N(\mu, \sigma^2/n)$ para $n$ grande — la distribución de la media siempre tiende a normal |

**Próxima sesión (4B):** Verificar la LGN y el TLC con datos reales
de la estación meteorológica ClimaLab.